In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("TFMX-MAR-08 -15 Corte de cobranza 2026.xlsx", sheet_name="TOTALES")

In [3]:
idxs = df.dropna().index.to_list() + [len(df)]
headers = df.dropna().iloc[0, :].apply(lambda x : x.upper().strip().replace(" ", "_")).to_list()
values = list()

for i in range(1, len(idxs)):
    chunk = df.iloc[idxs[i - 1]:idxs[i], :].copy()
    chunk.columns = headers
    chunk = chunk.iloc[1:]
    values.append(chunk)

In [4]:
test = values[0]
v = test[test["SAMPLE_FORM"].str.endswith("M.N.", na=False)]

In [5]:
v

,SAMPLE_ID,CODIGO,SAMPLE_FORM,SAMPLE_TYPE
2,03/15/2026,NaN,$ ...,NaN
7,"122 Analyses: Listeria monocytogenes MI , Sal...",TFM-015,122 X XXXXX Pesos M.N.,NaN
10,NaN,Subtotal =,Pesos M.N.,US Dollar
11,NaN,16% IVA =,Pesos M.N.,US Dollar
12,NaN,Total Amount Due =,Pesos M.N.,US Dollar


In [24]:
test[test["SAMPLE_FORM"].str.lower().str.strip() == "subtotal"].index

Index([8], dtype='int64')

In [6]:
prices = {'010-CCW': 250, '008-CCS': 150, '009-CCP': 150, '001-FPI': 38, '002-FPC': 36, '003-FPP': 80, '004-WWS': 42, '005-INS': 21,
          '006-LSH': 50, '007-LSG': 26, '011-LIS': 35, '012-PCT': 75, '013-SPC': 35, '014-MIG': 10, '015-RPP': 80, '016-SAL': 35,
          '017-IRP': 35, '018-LIS': 75, '019-IRL': 35, '020-PWT': 30, '002-FPC': 36, '001-GAP': 13, '002-GAP': 14, '021-AGT': 25,
          'TFM-001': 80, 'TFM-003': 22, 'TFM-008': 32, 'TFM-009': 21, 'TFM-010': 38, 'TFM-011': 22, 'TFM-012': 495, 'TFM-013': 21,
          'TFM-014': 36, 'GIP-001': 3000 / 20, 'CSS-001': 100, '003-GAP': 23, '021-LS6': 85, 'TFM-015': 160, 'TPA-015': 28}
        

In [7]:
x = v[1:-3].copy()
x

,SAMPLE_ID,CODIGO,SAMPLE_FORM,SAMPLE_TYPE
7,"122 Analyses: Listeria monocytogenes MI , Sal...",TFM-015,122 X XXXXX Pesos M.N.,NaN


In [8]:
dollar_price = 17.9218

x["CODE_PRICE"] = x["CODIGO"].map(prices)
x["SAMPLES"] = x["SAMPLE_FORM"].str.extract(r"(\d+)").astype(float)
x["TOTAL"] = x["CODE_PRICE"] * x["SAMPLES"] * dollar_price

In [9]:
subt = x["TOTAL"].sum()

In [10]:
x = v[-3:].copy()
x

,SAMPLE_ID,CODIGO,SAMPLE_FORM,SAMPLE_TYPE
10,NaN,Subtotal =,Pesos M.N.,US Dollar
11,NaN,16% IVA =,Pesos M.N.,US Dollar
12,NaN,Total Amount Due =,Pesos M.N.,US Dollar


In [11]:
x["MXN"] = [subt, subt * 0.16, subt * 1.16]
x["USD"] = [subt / dollar_price, (subt / dollar_price) * 0.16, (subt / dollar_price) * 1.16]
x

,SAMPLE_ID,CODIGO,SAMPLE_FORM,SAMPLE_TYPE,MXN,USD
10,NaN,Subtotal =,Pesos M.N.,US Dollar,349833.53600,19520.0
11,NaN,16% IVA =,Pesos M.N.,US Dollar,55973.36576,3123.2
12,NaN,Total Amount Due =,Pesos M.N.,US Dollar,405806.90176,22643.2
